# Customer Support — Priority Classification

## Step 1: Problem Definition
Predict ticket **priority** (Urgent, High, Medium, Low) from text and metadata. Supports automatic triage and SLA-aware routing.

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)

## Step 2: Data Collection
Load tickets from PostgreSQL (CSV fallback).

In [ ]:
from src.data_loader import load_tickets

raw_df = load_tickets(use_db=True)
raw_df.head()

## Step 3: Data Cleaning
Remove IDs and leakage columns; standardize column names.

In [ ]:
from src.preprocessor import clean_data

cleaned_df = clean_data(raw_df, task_type='classification')
print('Shape:', cleaned_df.shape)
cleaned_df.isnull().sum().sort_values(ascending=False).head(10)

### EDA: Priority distribution
See also `docs/eda/` for saved plots.

In [ ]:
cleaned_df['priority'].value_counts().plot(kind='bar', title='Priority Distribution')
plt.ylabel('Count')
plt.show()

## Step 4: Feature Engineering
Text length, word count, and date-derived features. TF-IDF is applied inside the sklearn pipeline.

In [ ]:
from src.preprocessor import engineer_features

fe_df = engineer_features(cleaned_df, task_type='classification')
fe_df[['desc_length', 'desc_word_count', 'priority']].describe()

## Step 5: Train-Test Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.label_engineering import derive_priority_label
from src.preprocessor import clean_data, engineer_features

RANDOM_SEED = 42
df_cls = engineer_features(clean_data(raw_df, 'classification'), 'classification')
df_cls = df_cls.sample(min(30000, len(df_cls)), random_state=RANDOM_SEED)
df_cls['priority'] = derive_priority_label(df_cls)
X = df_cls.drop(columns=['priority'])
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].fillna('Unknown')
y = df_cls['priority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## Step 6: Model Selection
Compare Logistic Regression, Random Forest, Gradient Boosting, and MLP (neural network) with TF-IDF + tabular features.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from src.model_trainer import get_classification_pipelines

results = {}
for name, pipe in get_classification_pipelines(X_train).items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[name] = {'f1': f1_score(y_test, pred, average='weighted'), 'acc': accuracy_score(y_test, pred)}
    print(f"{name}: F1={results[name]['f1']:.4f}  Acc={results[name]['acc']:.4f}")

best_name = max(results, key=lambda k: results[k]['f1'])
print('\nBest:', best_name)

## Step 7: Model Training
Train the best pipeline on the full training split.

In [ ]:
clf = get_classification_pipelines(X_train)[best_name]
clf.fit(X_train, y_train)
print('Training complete.')

## Step 8: Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=clf.classes_).plot()
plt.title('Confusion Matrix')
plt.show()

## Step 9: Model Tuning
Use GridSearchCV on the best model type (optional — run for final submission).

In [ ]:
from sklearn.model_selection import GridSearchCV

# Example tuning for Random Forest
param_grid = {'classifier__n_estimators': [100, 150], 'classifier__max_depth': [None, 20]}
grid = GridSearchCV(clf, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
# grid.fit(X_train, y_train)  # Uncomment for full tuning
print('Tuning grid defined. Uncomment grid.fit to run.')

## Step 10: Deployment
Save model bundle for Streamlit.

In [ ]:
import joblib
from src.paths import get_models_dir

acc = accuracy_score(y_test, y_pred)
bundle = {'model': clf, 'model_name': best_name, 'accuracy': acc}
path = os.path.join(get_models_dir(), 'classification_model.pkl')
joblib.dump(bundle, path)
print(f'Saved {path}  Accuracy={acc:.4f}')